# Training Analysis

Analyse a completed Forge training run: loss curves, LoRA weight statistics,
checkpoint comparison, and adapter merge verification.

**Prerequisites:** A completed training run with artifacts in `artifacts/training/<run>/`.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from forge.utils.config import load_training_config, load_yaml_config  # noqa: E402

print(f"Project root: {ROOT}")

## 1. Load Training Config

Inspect the training configuration that was used (or will be used) for the run.

In [ ]:
# Change this to match your training config
CONFIG_PATH = ROOT / "configs" / "models" / "turkcell_7b.yaml"

raw_config = load_yaml_config(CONFIG_PATH)
config = load_training_config(CONFIG_PATH)

print("=== Model ===")
print(f"  Name: {config.model.name}")
print(f"  Max seq length: {config.model.max_seq_length}")
print(f"  Dtype: {config.model.dtype}")

print("\n=== LoRA ===")
print(f"  r: {config.lora.r}")
print(f"  alpha: {config.lora.alpha}")
print(f"  dropout: {config.lora.dropout}")
print(f"  target_modules: {config.lora.target_modules}")

print("\n=== Training ===")
print(f"  Epochs: {config.training.num_epochs}")
print(f"  Batch size: {config.training.per_device_train_batch_size}")
print(f"  Learning rate: {config.training.learning_rate}")
print(f"  Grad accum: {config.training.gradient_accumulation_steps}")
print(f"  FP16: {config.training.fp16} | BF16: {config.training.bf16}")
print(f"  Scheduler: {config.training.lr_scheduler_type}")

print("\n=== Quantization ===")
print(f"  4-bit: {config.quantization.load_in_4bit}")
print(f"  Quant type: {config.quantization.bnb_4bit_quant_type}")
print(f"  Compute dtype: {config.quantization.bnb_4bit_compute_dtype}")

## 2. Load Training Logs

Parse the `trainer_state.json` saved by HuggingFace Trainer to extract
per-step loss and evaluation metrics.

In [ ]:
# Update this to your actual run directory
RUN_DIR = ROOT / "artifacts" / "training" / "forge-run"

trainer_state_path = RUN_DIR / "trainer_state.json"

if trainer_state_path.exists():
    with open(trainer_state_path) as f:
        trainer_state = json.load(f)

    log_history = trainer_state.get("log_history", [])
    print(f"Loaded {len(log_history)} log entries")
    best = trainer_state.get("best_metric", "N/A")
    print(f"Best metric: {best}")
    ckpt = trainer_state.get("best_model_checkpoint", "N/A")
    print(f"Best checkpoint: {ckpt}")
    steps = trainer_state.get("global_step", "N/A")
    print(f"Total steps: {steps}")
else:
    print(f"trainer_state.json not found at {trainer_state_path}")
    print("Run a training job first, or update RUN_DIR.")
    log_history = []

## 3. Loss Curves

Plot training loss and evaluation loss over training steps.

In [ ]:
train_steps = []
train_losses = []
eval_steps = []
eval_losses = []

for entry in log_history:
    step = entry.get("step", 0)
    if "loss" in entry:
        train_steps.append(step)
        train_losses.append(entry["loss"])
    if "eval_loss" in entry:
        eval_steps.append(step)
        eval_losses.append(entry["eval_loss"])

if train_losses:
    print(f"Training: {len(train_losses)} logged steps")
    print(f"  First loss: {train_losses[0]:.4f}")
    print(f"  Final loss: {train_losses[-1]:.4f}")
    print(f"  Min loss:   {min(train_losses):.4f}")

if eval_losses:
    best_eval = min(eval_losses)
    best_step = eval_steps[eval_losses.index(best_eval)]
    print(f"\nEval: {len(eval_losses)} checkpoints")
    print(f"  Best eval loss: {best_eval:.4f} (step {best_step})")

try:
    import matplotlib.pyplot as plt

    if train_losses:
        fig, ax = plt.subplots(1, 1, figsize=(12, 5))
        ax.plot(
            train_steps, train_losses,
            label="Training loss", alpha=0.7, color="#2196F3",
        )
        if eval_losses:
            ax.plot(
                eval_steps, eval_losses, "o-",
                label="Eval loss", color="#F44336", markersize=4,
            )
        ax.set_xlabel("Step")
        ax.set_ylabel("Loss")
        ax.set_title("Training & Evaluation Loss")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        print("No training data to plot.")
except ImportError:
    print("matplotlib not installed -- skipping plot.")

## 4. Learning Rate Schedule

Visualise the learning rate warmup and cosine decay schedule.

In [ ]:
lr_steps = []
lr_values = []

for entry in log_history:
    if "learning_rate" in entry:
        lr_steps.append(entry.get("step", 0))
        lr_values.append(entry["learning_rate"])

try:
    import matplotlib.pyplot as plt

    if lr_values:
        sched = config.training.lr_scheduler_type
        warmup = config.training.warmup_ratio
        fig, ax = plt.subplots(1, 1, figsize=(12, 4))
        ax.plot(lr_steps, lr_values, color="#4CAF50")
        ax.set_xlabel("Step")
        ax.set_ylabel("Learning Rate")
        ax.set_title(f"LR Schedule ({sched}, warmup={warmup})")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        print("No learning rate data to plot.")
except ImportError:
    print("matplotlib not installed -- skipping plot.")

## 5. LoRA Adapter Analysis

Load the saved LoRA adapter and analyse weight statistics to detect
potential issues (e.g., dead adapters, exploding weights).

In [ ]:
adapter_dir = RUN_DIR / "final"

# Look for adapter safetensors or bin files
adapter_files = (
    list(adapter_dir.glob("adapter_model.safetensors"))
    + list(adapter_dir.glob("adapter_model.bin"))
)

if adapter_files:
    print(f"Found adapter: {adapter_files[0]}")
    size_mb = adapter_files[0].stat().st_size / 1024 / 1024
    print(f"Size: {size_mb:.1f} MB")

    # Load adapter config
    adapter_config_path = adapter_dir / "adapter_config.json"
    if adapter_config_path.exists():
        with open(adapter_config_path) as f:
            adapter_config = json.load(f)
        print("\nAdapter Config:")
        keys = ("r", "lora_alpha", "lora_dropout", "target_modules", "task_type", "bias")
        for k in keys:
            if k in adapter_config:
                print(f"  {k}: {adapter_config[k]}")
else:
    print(f"No adapter found at {adapter_dir}")
    print("Run training first or update RUN_DIR.")

In [ ]:
# Weight statistics (requires torch)
try:
    import torch

    if adapter_files:
        if str(adapter_files[0]).endswith(".safetensors"):
            from safetensors.torch import load_file
            weights = load_file(str(adapter_files[0]))
        else:
            weights = torch.load(str(adapter_files[0]), map_location="cpu")

        print(f"\nAdapter weight tensors: {len(weights)}")
        header = (
            f"{'Layer':<50} {'Shape':<20} "
            f"{'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}"
        )
        print(header)
        print("-" * len(header))

        for name, tensor in sorted(weights.items()):
            t = tensor.float()
            shape = str(list(tensor.shape))
            print(
                f"{name:<50} {shape:<20} "
                f"{t.mean().item():>10.6f} {t.std().item():>10.6f} "
                f"{t.min().item():>10.6f} {t.max().item():>10.6f}"
            )

        # Check for dead adapters (all zeros)
        dead = sum(
            1 for t in weights.values()
            if t.float().abs().max().item() < 1e-8
        )
        if dead:
            print(f"\n!! {dead} adapter(s) appear dead (all near zero)")
        else:
            print(f"\nAll {len(weights)} adapters have non-trivial weights.")
except ImportError:
    print("torch not installed -- skipping weight analysis.")

## 6. Evaluation Results

Load and display benchmark results from `artifacts/eval/results.json`.

In [ ]:
EVAL_DIR = ROOT / "artifacts" / "eval"
results_path = EVAL_DIR / "results.json"

if results_path.exists():
    with open(results_path) as f:
        eval_data = json.load(f)

    ts = eval_data.get("timestamp", "N/A")
    print(f"Evaluation timestamp: {ts}")
    summary = eval_data.get("summary", {})
    p = summary.get("passed", 0)
    fa = summary.get("failed", 0)
    print(f"Benchmarks: {summary.get('total', 0)} total, {p} passed, {fa} failed")

    header = f"{'Benchmark':<20} {'Score':>10} {'Status':>8} {'Duration':>10}"
    print(f"\n{header}")
    print("-" * len(header))
    for bench in eval_data.get("benchmarks", []):
        status = "PASS" if bench["passed"] else "FAIL"
        dur = bench["duration_seconds"]
        print(f"{bench['name']:<20} {bench['score']:>10.4f} {status:>8} {dur:>9.1f}s")
else:
    print(f"No eval results found at {results_path}")
    print("Run: uv run python scripts/run_eval.py --model <path>")

## 7. WandB Integration (Optional)

If WandB was enabled during training, fetch run data for comparison.

In [ ]:
try:
    import wandb

    api = wandb.Api()
    project = config.wandb.project
    runs = api.runs(f"{api.default_entity}/{project}", per_page=5)

    print(f"Recent runs in '{project}':")
    header = f"{'Name':<30} {'State':<12} {'Duration':>10} {'Final Loss':>12}"
    print(header)
    print("-" * len(header))
    for run in runs:
        duration = f"{run.summary.get('_runtime', 0) / 60:.1f}m"
        loss = run.summary.get("train/loss", "N/A")
        loss_str = f"{loss:.4f}" if isinstance(loss, (int, float)) else str(loss)
        print(f"{run.name:<30} {run.state:<12} {duration:>10} {loss_str:>12}")
except ImportError:
    print("wandb not installed. Install with: pip install wandb")
except Exception as exc:
    print(f"Could not connect to WandB: {exc}")

## 8. Merged Model Verification

Check if a merged model exists and verify its metadata.

In [ ]:
MERGED_DIR = ROOT / "artifacts" / "merged"

if MERGED_DIR.exists():
    merged_models = [d for d in MERGED_DIR.iterdir() if d.is_dir()]
    print(f"Found {len(merged_models)} merged model(s):")

    for model_dir in merged_models:
        print(f"\n  {model_dir.name}/")
        merge_info_path = model_dir / "merge_info.json"
        if merge_info_path.exists():
            with open(merge_info_path) as f:
                info = json.load(f)
            print(f"    Base model:  {info.get('base_model', 'N/A')}")
            print(f"    Adapter:     {info.get('adapter_path', 'N/A')}")
            print(f"    Method:      {info.get('merge_method', 'N/A')}")

        # Check tokenizer config
        tok_path = model_dir / "tokenizer_config.json"
        if tok_path.exists():
            with open(tok_path) as f:
                tok_config = json.load(f)
            tok_class = tok_config.get("tokenizer_class", "N/A")
            print(f"    Tokenizer:   {tok_class}")
            if tok_class == "TokenizersBackend":
                print("    !! Needs patching for vLLM")

        # Model size
        total_bytes = sum(
            f.stat().st_size for f in model_dir.rglob("*") if f.is_file()
        )
        size_gb = total_bytes / 1024 / 1024 / 1024
        print(f"    Total size:  {size_gb:.2f} GB")
else:
    print("No merged models found.")
    print("Run merge_and_push.py to create one.")

## Summary

This notebook covered:
1. Training config inspection (model, LoRA, quantization)
2. Loss curve analysis (train + eval)
3. Learning rate schedule visualisation
4. LoRA adapter weight statistics
5. Evaluation benchmark results
6. WandB run comparison
7. Merged model verification

**Key things to watch for:**
- Eval loss should decrease or plateau (not increase = overfitting)
- All LoRA adapters should have non-trivial weights (no dead adapters)
- Tokenizer class should be `LlamaTokenizer` (not `TokenizersBackend`) for vLLM
- All evaluation benchmarks should pass their thresholds